# Warmup plus cosine decay with LambdaLR

Transformer training almost always ramps the learning rate up linearly, then decays it along a cosine to a floor. Implement that schedule as a `torch.optim.lr_scheduler.LambdaLR` over an optimizer with several parameter groups.

## The multiplier
For update $s = 0, 1, 2, \ldots$ with warmup length $w \ge 0$, total length $S$ and floor ratio $\rho \in [0, 1]$:

$$\text{factor}(s) = \begin{cases} \dfrac{s+1}{w} & s < w \\[2mm] \rho + (1-\rho)\,\tfrac12\big(1 + \cos(\pi\, p)\big), \quad p = \min\!\Big(1,\ \dfrac{s - w}{\max(1,\ S - w)}\Big) & s \ge w \end{cases}$$

So the first update already uses $1/w$ of the peak (not 0), the peak is reached at update $w-1$ or $w$, and after update $S$ the multiplier stays at $\rho$. With $w = 0$ there is no warmup.

## What's graded
Write `warmup_cosine_lrs(base_lrs, warmup, total, min_ratio, steps)`:

- `base_lrs`: list of floats, one per parameter group (each group may hold a single dummy parameter).
- Build a `torch.optim.SGD` (or AdamW) with one group per base lr, wrap it in `LambdaLR` with your multiplier, and run `steps` updates: read every group's lr, then `optimizer.step()`, then `scheduler.step()`.
- Return a float64 tensor `(steps, G)`: row `s` holds the learning rate of each group at update `s`.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import math
import torch

def warmup_cosine_lrs(base_lrs, warmup, total, min_ratio, steps):
    """Return (steps, G) float64: lr of each param group at each update."""
    params = [torch.zeros(1, requires_grad=True) for _ in base_lrs]
    opt = torch.optim.SGD([{'params': [p], 'lr': lr} for p, lr in zip(params, base_lrs)])

    def factor(s):
        return 1.0          # TODO: warmup, then cosine down to min_ratio

    sched = torch.optim.lr_scheduler.LambdaLR(opt, factor)
    rows = []
    for _ in range(steps):
        rows.append([g['lr'] for g in opt.param_groups])
        opt.step()
        sched.step()
    return torch.tensor(rows, dtype=torch.float64)


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [[1.0], 2, 6, 0.0, 8], "e": {"__nd__": [[0.5], [1.0], [1.0], [0.8535533905932737], [0.5], [0.14644660940672627], [0.0], [0.0]], "dtype": "float64"}}, {"a": [[0.1, 0.01], 0, 4, 0.1, 6], "e": {"__nd__": [[0.1, 0.01], [0.08681980515339464, 0.008681980515339464], [0.05500000000000001, 0.0055000000000000005], [0.023180194846605367, 0.0023180194846605367], [0.010000000000000002, 0.001], [0.010000000000000002, 0.001]], "dtype": "float64"}}, {"a": [[0.0003], 5, 20, 0.1, 25], "e": {"__nd__": [[5.9999999999999995e-05], [0.00011999999999999999], [0.00017999999999999998], [0.00023999999999999998], [0.0003], [0.0003], [0.0002970499260990637], [0.0002883286367817511], [0.00027421729424061787], [0.00025533263185844587], [0.00023249999999999999], [0.00020671729424061788], [0.0001791113425411332], [0.0001508886574588668], [0.00012328270575938212], [9.750000000000003e-05], [7.466736814155418e-05], [5.578270575938211e-05], [4.167136321824887e-05], [3.2950073900936234e-05], [2.9999999999999997e-05], [2.9999999999999997e-05], [2.9999999999999997e-05], [2.9999999999999997e-05], [2.9999999999999997e-05]], "dtype": "float64"}}, {"a": [[1.0, 2.0, 0.5], 3, 3, 0.2, 6], "e": {"__nd__": [[0.3333333333333333, 0.6666666666666666, 0.16666666666666666], [0.6666666666666666, 1.3333333333333333, 0.3333333333333333], [1.0, 2.0, 0.5], [1.0, 2.0, 0.5], [0.2, 0.4, 0.1], [0.2, 0.4, 0.1]], "dtype": "float64"}}, {"a": [[1.0], 1, 2, 0.5, 4], "e": {"__nd__": [[1.0], [1.0], [0.5], [0.5]], "dtype": "float64"}}, {"a": [[0.0006, 6e-05], 10, 100, 0.0, 120], "e": {"__nd__": [[5.9999999999999995e-05, 6e-06], [0.00011999999999999999, 1.2e-05], [0.00017999999999999998, 1.8e-05], [0.00023999999999999998, 2.4e-05], [0.0003, 3e-05], [0.00035999999999999997, 3.6e-05], [0.00041999999999999996, 4.2e-05], [0.00047999999999999996, 4.8e-05], [0.00054, 5.4000000000000005e-05], [0.0006, 6e-05], [0.0006, 6e-05], [0.0005998172481057287, 5.9981724810572876e-05], [0.0005992692150779472, 5.992692150779473e-05], [0.0005983565686104819, 5.98356568610482e-05], [0.000597080420622471, 5.970804206224711e-05], [0.0005954423259036624, 5.954423259036624e-05], [0.0005934442802201416, 5.934442802201417e-05], [0.0005910887178827988, 5.910887178827989e-05], [0.0005883785087814956, 5.883785087814957e-05], [0.0005853169548885461, 5.853169548885461e-05], [0.0005819077862357724, 5.819077862357725e-05], [0.0005781551563700362, 5.781551563700362e-05], [0.0005740636372927802, 5.7406363729278026e-05], [0.00056963821388975, 5.6963821388975015e-05], [0.000564884277857678, 5.648842778576781e-05], [0.0005598076211353316, 5.5980762113533166e-05], [0.0005544144288469277, 5.544144288469278e-05], [0.0005487112717665124, 5.487112717665125e-05], [0.0005427050983124842, 5.427050983124843e-05], [0.0005364032260820165, 5.3640322608201656e-05], [0.0005298133329356933, 5.298133329356934e-05], [0.0005229434476432183, 5.2294344764321825e-05], [0.0005158019401015953, 5.158019401015953e-05], [0.0005083975111376992, 5.083975111376992e-05], [0.0005007391819076574, 5.007391819076575e-05], [0.0004928362829059618, 4.928362829059618e-05], [0.00048469844259769747, 4.846984425976975e-05], [0.0004763355756877419, 4.7633557568774194e-05], [0.00046775787104122397, 4.6775787104122404e-05], [0.0004589757792699614, 4.589757792699615e-05], [0.00045, 4.5e-05], [0.00044084146883576717, 4.4084146883576726e-05], [0.0004315113440367232, 4.315113440367232e-05], [0.00042202099292274007, 4.2202099292274015e-05], [0.0004123819780247736, 4.123819780247737e-05], [0.00040260604299770063, 4.0260604299770066e-05], [0.0003927050983124842, 3.927050983124842e-05], [0.0003826912067450997, 3.826912067450997e-05], [0.00037257656867990027, 3.725765686799003e-05], [0.0003623735072453277, 3.6237350724532775e-05], [0.0003520944533000791, 3.520944533000792e-05], [0.0003417519302880196, 3.4175193028801965e-05], [0.000331358538980296, 3.313585389802961e-05], [0.0003209269421232375, 3.209269421232376e-05], [0.00031046984901075033, 3.104698490107504e-05], [0.0003, 3e-05], [0.0002895301509892498, 2.895301509892498e-05], [0.00027907305787676237, 2.7907305787676243e-05], [0.000268641461019704, 2.6864146101970402e-05], [0.0002582480697119804, 2.5824806971198044e-05], [0.0002479055466999209, 2.479055466999209e-05], [0.00023762649275467222, 2.3762649275467226e-05], [0.0002274234313200997, 2.274234313200997e-05], [0.00021730879325490026, 2.1730879325490026e-05], [0.0002072949016875158, 2.072949016875158e-05], [0.00019739395700229937, 1.973939570022994e-05], [0.00018761802197522637, 1.8761802197522638e-05], [0.00017797900707725996, 1.7797900707726e-05], [0.00016848865596327673, 1.6848865596327676e-05], [0.00015915853116423283, 1.5915853116423285e-05], [0.00015000000000000004, 1.5000000000000007e-05], [0.00014102422073003856, 1.4102422073003856e-05], [0.00013224212895877598, 1.32242128958776e-05], [0.00012366442431225807, 1.2366442431225809e-05], [0.0001153015574023025, 1.1530155740230252e-05], [0.00010716371709403818, 1.0716371709403819e-05], [9.926081809234262e-05, 9.926081809234262e-06], [9.160248886230079e-05, 9.16024888623008e-06], [8.41980598984047e-05, 8.419805989840472e-06], [7.705655235678168e-05, 7.70565523567817e-06], [7.018666706430662e-05, 7.018666706430663e-06], [6.359677391798343e-05, 6.359677391798343e-06], [5.7294901687515795e-05, 5.72949016875158e-06], [5.128872823348751e-05, 5.128872823348752e-06], [4.558557115307221e-05, 4.558557115307222e-06], [4.019237886466838e-05, 4.019237886466839e-06], [3.51157221423219e-05, 3.5115722142321906e-06], [3.0361786110249954e-05, 3.036178611024996e-06], [2.5936362707219704e-05, 2.5936362707219708e-06], [2.1844843629963803e-05, 2.1844843629963808e-06], [1.8092213764227503e-05, 1.8092213764227506e-06], [1.468304511145394e-05, 1.4683045111453942e-06], [1.162149121850433e-05, 1.1621491218504332e-06], [8.911282117201057e-06, 8.911282117201058e-07], [6.555719779858293e-06, 6.555719779858294e-07], [4.557674096337593e-06, 4.557674096337594e-07], [2.919579377528891e-06, 2.9195793775288916e-07], [1.6434313895180128e-06, 1.6434313895180132e-07], [7.307849220527406e-07, 7.307849220527407e-08], [1.8275189427127136e-07, 1.8275189427127137e-08], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0], [0.0, 0.0]], "dtype": "float64"}}, {"a": [[1.0], 4, 10, 1.0, 12], "e": {"__nd__": [[0.25], [0.5], [0.75], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0], [1.0]], "dtype": "float64"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(warmup_cosine_lrs, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")